In [1]:
!pip install nltk gensim

In [2]:
import nltk
nltk.download('brown')
nltk.download('punkt')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
from nltk.corpus import brown
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import multiprocessing

In [4]:
documents = []
labels = []

for fileid in brown.fileids():
    category = brown.categories(fileid)[0]
    sents = brown.sents(fileid)

    for sent in sents:
        documents.append([w.lower() for w in sent])
        labels.append(category)

print("Total Documents:", len(documents))

Total Documents: 57340


In [5]:
tagged_data = [
    TaggedDocument(words=doc, tags=[str(i)])
    for i, doc in enumerate(documents)
]

In [6]:
cores = multiprocessing.cpu_count()

model_dm = Doc2Vec(
    vector_size=200,
    window=8,
    min_count=2,
    workers=cores,
    epochs=30,
    dm=1,
    negative=5
)

model_dm.build_vocab(tagged_data)
model_dm.train(
    tagged_data,
    total_examples=model_dm.corpus_count,
    epochs=model_dm.epochs
)

In [7]:
model_dbow = Doc2Vec(
    vector_size=200,
    window=8,
    min_count=2,
    workers=cores,
    epochs=30,
    dm=0,
    negative=5
)

model_dbow.build_vocab(tagged_data)
model_dbow.train(
    tagged_data,
    total_examples=model_dbow.corpus_count,
    epochs=model_dbow.epochs
)

In [8]:
import numpy as np
import random

random.seed(42)
np.random.seed(42)

In [9]:
doc_vectors = []

for i in range(len(tagged_data)):
    vec_dm = model_dm.dv[str(i)]
    vec_dbow = model_dbow.dv[str(i)]

    combined = np.hstack((vec_dm, vec_dbow))
    doc_vectors.append(combined)

doc_vectors = np.array(doc_vectors)

print("Final Vector Shape:", doc_vectors.shape)

Final Vector Shape: (57340, 400)


In [10]:
unique_categories = list(set(labels))
category_to_id = {cat: i for i, cat in enumerate(unique_categories)}

y = np.array([category_to_id[c] for c in labels])

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [12]:
unique_categories = list(set(labels))
category_to_id = {cat: i for i, cat in enumerate(unique_categories)}

y = np.array([category_to_id[c] for c in labels])

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    doc_vectors, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
clf = SVC(kernel = 'rbf')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Classification Accuracy:", accuracy)

Classification Accuracy: 0.38228113010115106
